In [1]:
import os
import glob
import pandas as pd

print("=" * 70)
print("SEARCHING FOR DLL DATASET")
print("=" * 70)

# Search common Kaggle locations
search_paths = [
    "/kaggle/working/*.csv",
    "/kaggle/input/**/*.csv"
]

csv_files = []

for pattern in search_paths:
    csv_files.extend(glob.glob(pattern, recursive=True))

print(f"\nCSV files found: {len(csv_files)}")

for path in csv_files:
    try:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{path}  |  {size_mb:.2f} MB")
    except:
        pass

SEARCHING FOR DLL DATASET

CSV files found: 4
/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv  |  9.34 MB
/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv  |  37.31 MB
/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv  |  1235.97 MB
/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv  |  7.45 MB


In [2]:
import os
import glob
import pandas as pd

print("=" * 70)
print("IDENTIFYING DLL DATASET")
print("=" * 70)

csv_files = []

for pattern in [
    "/kaggle/working/*.csv",
    "/kaggle/input/**/*.csv"
]:
    csv_files.extend(glob.glob(pattern, recursive=True))

dll_dataset_path = None

for path in csv_files:

    try:
        # Read only header first
        cols = pd.read_csv(path, nrows=0).columns.tolist()

        # DLL dataset characteristics
        has_sha = "SHA256" in cols
        has_type = "Type" in cols

        # DLL-like columns
        dll_cols = [
            c for c in cols
            if c.lower().endswith(".dll") or c.lower().endswith(".drv")
        ]

        if has_sha and has_type and len(dll_cols) > 500:
            dll_dataset_path = path

            print("\nDLL DATASET FOUND!")
            print("Path:", path)
            print("Total columns:", len(cols))
            print("DLL columns:", len(dll_cols))

            break

    except Exception as e:
        continue

if dll_dataset_path is None:
    raise FileNotFoundError(
        "Could not automatically identify the DLL dataset."
    )

print("\nDataset path saved as:")
print(dll_dataset_path)

IDENTIFYING DLL DATASET

DLL DATASET FOUND!
Path: /kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv
Total columns: 631
DLL columns: 626

Dataset path saved as:
/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv


In [3]:
DLL_PATH = dll_dataset_path

df = pd.read_csv(DLL_PATH)

print("=" * 70)
print("DLL DATASET LOADED")
print("=" * 70)

print("Shape:", df.shape)
print("Columns:", len(df.columns))

print("\nFirst columns:")
print(df.columns[:10].tolist())

print("\nTarget distribution:")
print(df["Type"].value_counts())

DLL DATASET LOADED
Shape: (29498, 631)
Columns: 631

First columns:
['SHA256', 'Type', 'advapi32.dll', 'kernel32.dll', 'vspmsg.dll', 'ole32.dll', 'oleaut32.dll', 'psapi.dll', 'setupapi.dll', 'shlwapi.dll']

Target distribution:
Type
4    5076
1    5022
3    4957
2    4643
5    4224
6    3699
0    1877
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np

print("=" * 70)
print("DLL FREQUENCY ANALYSIS")
print("=" * 70)

# DLL columns = everything except metadata
META_COLS = ["SHA256", "Type"]
DLL_COLS = [c for c in df.columns if c not in META_COLS]

print("\nTotal DLL features:", len(DLL_COLS))

# Frequency = number of samples containing each DLL
dll_frequency = df[DLL_COLS].sum(axis=0)

# Percentage of dataset
dll_percentage = (dll_frequency / len(df)) * 100

freq_df = pd.DataFrame({
    "DLL": DLL_COLS,
    "Samples_With_DLL": dll_frequency.values,
    "Percentage": dll_percentage.values
})

freq_df = freq_df.sort_values(
    "Samples_With_DLL",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("FREQUENCY STATISTICS")
print("=" * 70)

print(freq_df["Samples_With_DLL"].describe())

print("\n" + "=" * 70)
print("FREQUENCY BUCKETS")
print("=" * 70)

thresholds = [1, 2, 5, 10, 20, 50, 100, 250, 500, 1000]

for threshold in thresholds:
    count = (freq_df["Samples_With_DLL"] >= threshold).sum()
    print(
        f"DLLs appearing in >= {threshold:4} samples: "
        f"{count:3}"
    )

print("\n" + "=" * 70)
print("CANDIDATE FREQUENCY THRESHOLDS")
print("=" * 70)

candidate_thresholds = [5, 10, 20, 50, 100]

for threshold in candidate_thresholds:

    selected = freq_df[
        freq_df["Samples_With_DLL"] >= threshold
    ]

    print(
        f"\nThreshold >= {threshold} samples"
    )
    print(
        f"Features retained : {len(selected)}"
    )
    print(
        f"Percentage retained: "
        f"{len(selected) / len(DLL_COLS) * 100:.2f}%"
    )
    print(
        f"Minimum frequency : "
        f"{selected['Samples_With_DLL'].min() if len(selected) else 0}"
    )
    print(
        f"Maximum frequency : "
        f"{selected['Samples_With_DLL'].max() if len(selected) else 0}"
    )

print("\n" + "=" * 70)
print("TOP 30 MOST COMMON DLLs")
print("=" * 70)

print(
    freq_df.head(30).to_string(index=False)
)

print("\n" + "=" * 70)
print("RAREST DLLs")
print("=" * 70)

print(
    freq_df.tail(30).to_string(index=False)
)

print("\n" + "=" * 70)
print("DLL FREQUENCY ANALYSIS COMPLETE")
print("=" * 70)

DLL FREQUENCY ANALYSIS

Total DLL features: 629

FREQUENCY STATISTICS
count      629.000000
mean       147.887122
std       1020.206412
min          1.000000
25%          1.000000
50%          2.000000
75%          8.000000
max      14598.000000
Name: Samples_With_DLL, dtype: float64

FREQUENCY BUCKETS
DLLs appearing in >=    1 samples: 629
DLLs appearing in >=    2 samples: 355
DLLs appearing in >=    5 samples: 198
DLLs appearing in >=   10 samples: 142
DLLs appearing in >=   20 samples: 118
DLLs appearing in >=   50 samples:  83
DLLs appearing in >=  100 samples:  51
DLLs appearing in >=  250 samples:  29
DLLs appearing in >=  500 samples:  20
DLLs appearing in >= 1000 samples:  16

CANDIDATE FREQUENCY THRESHOLDS

Threshold >= 5 samples
Features retained : 198
Percentage retained: 31.48%
Minimum frequency : 5
Maximum frequency : 14598

Threshold >= 10 samples
Features retained : 142
Percentage retained: 22.58%
Minimum frequency : 10
Maximum frequency : 14598

Threshold >= 20 samples

In [5]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np

print("=" * 70)
print("DLL MUTUAL INFORMATION FEATURE SELECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Frequency filtering
# ------------------------------------------------------------

FREQ_THRESHOLD = 20

selected_freq = freq_df[
    freq_df["Samples_With_DLL"] >= FREQ_THRESHOLD
].copy()

DLL_CANDIDATES = selected_freq["DLL"].tolist()

print("\nFrequency threshold:", FREQ_THRESHOLD)
print("Starting DLL features:", len(DLL_COLS))
print("After frequency filtering:", len(DLL_CANDIDATES))

# ------------------------------------------------------------
# 2. Prepare feature matrix
# ------------------------------------------------------------

X = df[DLL_CANDIDATES].astype(np.uint8)
y = df["Type"]

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

# ------------------------------------------------------------
# 3. Calculate Mutual Information
# ------------------------------------------------------------

print("\nCalculating Mutual Information...")
print("This may take a little while...")

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

# ------------------------------------------------------------
# 4. Create results table
# ------------------------------------------------------------

mi_df = pd.DataFrame({
    "DLL": DLL_CANDIDATES,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MUTUAL INFORMATION STATISTICS")
print("=" * 70)

print(mi_df["MI_Score"].describe())

print("\n" + "=" * 70)
print("TOP 30 DLLs BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(30).to_string(index=False)
)

print("\n" + "=" * 70)
print("BOTTOM 20 DLLs BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.tail(20).to_string(index=False)
)

# ------------------------------------------------------------
# 6. MI thresholds
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MI THRESHOLD ANALYSIS")
print("=" * 70)

mi_thresholds = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.03,
    0.05
]

for threshold in mi_thresholds:

    count = (
        mi_df["MI_Score"] >= threshold
    ).sum()

    print(
        f"MI >= {threshold:.3f}: "
        f"{count:3} DLLs"
    )

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

MI_PATH = "/kaggle/working/DLL_Candidate_Features_MI.csv"

mi_df.to_csv(
    MI_PATH,
    index=False
)

print("\n" + "=" * 70)
print("DLL MI ANALYSIS COMPLETE")
print("=" * 70)

print("\nSaved to:")
print(MI_PATH)

DLL MUTUAL INFORMATION FEATURE SELECTION

Frequency threshold: 20
Starting DLL features: 629
After frequency filtering: 118

Feature matrix shape: (29498, 118)
Target shape: (29498,)

Calculating Mutual Information...
This may take a little while...

MUTUAL INFORMATION STATISTICS
count    118.000000
mean       0.017615
std        0.043130
min        0.001006
25%        0.002743
50%        0.003955
75%        0.013580
max        0.336208
Name: MI_Score, dtype: float64

TOP 30 DLLs BY MUTUAL INFORMATION
                  DLL  MI_Score
          mscoree.dll  0.336208
         kernel32.dll  0.220688
         msvbvm60.dll  0.155394
           user32.dll  0.152095
            gdi32.dll  0.116904
            winmm.dll  0.072743
            mfc42.dll  0.060209
         advapi32.dll  0.054260
          shlwapi.dll  0.052860
           msvcrt.dll  0.052643
         oleaut32.dll  0.052536
         comdlg32.dll  0.041725
         winspool.drv  0.032078
           oledlg.dll  0.029378
         shfo

In [6]:
import pandas as pd
import numpy as np

print("=" * 70)
print("DLL REDUNDANCY / CORRELATION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load MI results
# ------------------------------------------------------------

MI_PATH = "/kaggle/working/DLL_Candidate_Features_MI.csv"

mi_df = pd.read_csv(MI_PATH)

# ------------------------------------------------------------
# 2. Select MI-qualified DLLs
# ------------------------------------------------------------

MI_THRESHOLD = 0.010

candidate_df = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

candidate_df = candidate_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

DLL_CANDIDATES = candidate_df["DLL"].tolist()

print("\nMI threshold:", MI_THRESHOLD)
print("Starting DLLs:", len(mi_df))
print("MI-qualified DLLs:", len(DLL_CANDIDATES))

print("\nCandidate DLLs:")
print(candidate_df.to_string(index=False))

# ------------------------------------------------------------
# 3. Create feature matrix
# ------------------------------------------------------------

X_dll = df[DLL_CANDIDATES].astype(np.uint8)

print("\nFeature matrix:", X_dll.shape)

# ------------------------------------------------------------
# 4. Correlation matrix
# ------------------------------------------------------------

print("\nCalculating correlation matrix...")

corr_matrix = X_dll.corr()

print("Correlation matrix created.")
print("Shape:", corr_matrix.shape)

# ------------------------------------------------------------
# 5. Extract upper triangle
# ------------------------------------------------------------

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_thresholds = [0.95, 0.90, 0.85, 0.80]

print("\n" + "=" * 70)
print("CORRELATION THRESHOLD SUMMARY")
print("=" * 70)

for threshold in correlation_thresholds:

    pairs = (
        upper.abs() >= threshold
    ).sum().sum()

    print(
        f"|Correlation| >= {threshold:.2f}: "
        f"{int(pairs)} pairs"
    )

# ------------------------------------------------------------
# 6. Top correlated pairs
# ------------------------------------------------------------

threshold = 0.90

pairs = []

for feature1 in upper.columns:

    for feature2 in upper.index:

        value = upper.loc[feature2, feature1]

        if pd.notna(value) and abs(value) >= threshold:

            pairs.append({
                "DLL_1": feature2,
                "DLL_2": feature1,
                "Correlation": value
            })

pairs_df = pd.DataFrame(pairs)

if len(pairs_df) > 0:

    pairs_df = pairs_df.sort_values(
        "Correlation",
        key=lambda x: x.abs(),
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 70)
    print("HIGHLY CORRELATED DLL PAIRS")
    print("=" * 70)

    print(
        pairs_df.head(50).to_string(index=False)
    )

else:

    print("\nNo highly correlated pairs found.")

# ------------------------------------------------------------
# 7. Save correlation pairs
# ------------------------------------------------------------

CORR_PATH = "/kaggle/working/DLL_Correlation_Pairs.csv"

pairs_df.to_csv(
    CORR_PATH,
    index=False
)

print("\nSaved correlation results to:")
print(CORR_PATH)

print("\n" + "=" * 70)
print("DLL REDUNDANCY ANALYSIS COMPLETE")
print("=" * 70)

DLL REDUNDANCY / CORRELATION ANALYSIS

MI threshold: 0.01
Starting DLLs: 118
MI-qualified DLLs: 34

Candidate DLLs:
                     DLL  MI_Score
             mscoree.dll  0.336208
            kernel32.dll  0.220688
            msvbvm60.dll  0.155394
              user32.dll  0.152095
               gdi32.dll  0.116904
               winmm.dll  0.072743
               mfc42.dll  0.060209
            advapi32.dll  0.054260
             shlwapi.dll  0.052860
              msvcrt.dll  0.052643
            oleaut32.dll  0.052536
            comdlg32.dll  0.041725
            winspool.drv  0.032078
              oledlg.dll  0.029378
            shfolder.dll  0.025432
             winhttp.dll  0.025303
              oleacc.dll  0.023286
             version.dll  0.023075
            imagehlp.dll  0.021162
             gdiplus.dll  0.018735
           libintl-8.dll  0.017190
             msvcr90.dll  0.017095
       libglib-2.0-0.dll  0.016900
             msimg32.dll  0.016655
         

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

print("=" * 70)
print("DLL MI + CORRELATION-REDUCED FEATURE SELECTION")
print("=" * 70)

# ============================================================
# PATH
# ============================================================

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading DLL dataset...")

df = pd.read_csv(DLL_PATH)

print("Dataset shape:", df.shape)

# First two columns are SHA256 and Type
target = df["Type"]

dll_features = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

print("Total DLL features:", len(dll_features))

# ============================================================
# STEP 1 — FREQUENCY FILTER
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: FREQUENCY FILTERING")
print("=" * 70)

X_all = df[dll_features].astype(np.uint8)

# Number of samples in which each DLL occurs
dll_frequency = X_all.sum(axis=0)

FREQUENCY_THRESHOLD = 20

qualified_frequency = dll_frequency[
    dll_frequency >= FREQUENCY_THRESHOLD
].index.tolist()

print(
    f"\nFrequency threshold: >= {FREQUENCY_THRESHOLD}"
)

print(
    "DLLs before filtering:",
    len(dll_features)
)

print(
    "DLLs after frequency filtering:",
    len(qualified_frequency)
)

# ============================================================
# STEP 2 — MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: MUTUAL INFORMATION")
print("=" * 70)

X_freq = df[qualified_frequency].astype(np.uint8)

print("Feature matrix:", X_freq.shape)
print("Target:", target.shape)

print("\nCalculating Mutual Information...")

mi_scores = mutual_info_classif(
    X_freq,
    target,
    discrete_features=True,
    random_state=42
)

mi_df = pd.DataFrame({
    "DLL": qualified_frequency,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# SAVE MI RESULTS
# ============================================================

MI_PATH = "/kaggle/working/DLL_Candidate_Features_MI.csv"

mi_df.to_csv(
    MI_PATH,
    index=False
)

print("\nMI results saved to:")
print(MI_PATH)

# ============================================================
# MI FILTER
# ============================================================

MI_THRESHOLD = 0.01

mi_qualified = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

print("\n" + "=" * 70)
print("STEP 3: MUTUAL INFORMATION FILTER")
print("=" * 70)

print("MI threshold:", MI_THRESHOLD)
print("Before MI filtering:", len(mi_df))
print("After MI filtering:", len(mi_qualified))

print("\nMI-qualified DLLs:")
print(
    mi_qualified.to_string(index=False)
)

# ============================================================
# STEP 4 — CORRELATION ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: CORRELATION ANALYSIS")
print("=" * 70)

candidate_dlls = mi_qualified["DLL"].tolist()

X_mi = df[candidate_dlls].astype(np.uint8)

print(
    "\nCorrelation feature matrix:",
    X_mi.shape
)

corr = X_mi.corr()

CORR_THRESHOLD = 0.90

print(
    f"Correlation threshold: |r| >= {CORR_THRESHOLD}"
)

# ============================================================
# ITERATIVE REDUNDANCY REDUCTION
# ============================================================

remaining = mi_qualified.copy()

removed_records = []

while True:

    features = remaining["DLL"].tolist()

    if len(features) <= 1:
        break

    current_corr = X_mi[features].corr()

    # Upper triangular matrix only
    upper = current_corr.where(
        np.triu(
            np.ones(current_corr.shape),
            k=1
        ).astype(bool)
    )

    # Find correlated pairs
    pairs = []

    for col in upper.columns:

        for row in upper.index:

            value = upper.loc[row, col]

            if (
                pd.notna(value)
                and abs(value) >= CORR_THRESHOLD
            ):
                pairs.append(
                    (row, col, value)
                )

    # No more redundant pairs
    if not pairs:
        break

    # Strongest correlation first
    row, col, correlation = max(
        pairs,
        key=lambda x: abs(x[2])
    )

    row_mi = remaining.loc[
        remaining["DLL"] == row,
        "MI_Score"
    ].iloc[0]

    col_mi = remaining.loc[
        remaining["DLL"] == col,
        "MI_Score"
    ].iloc[0]

    # Keep higher MI
    if row_mi >= col_mi:

        kept = row
        removed = col

        kept_mi = row_mi
        removed_mi = col_mi

    else:

        kept = col
        removed = row

        kept_mi = col_mi
        removed_mi = row_mi

    removed_records.append({
        "Removed": removed,
        "Kept": kept,
        "Correlation": correlation,
        "Removed_MI": removed_mi,
        "Kept_MI": kept_mi
    })

    remaining = remaining[
        remaining["DLL"] != removed
    ].reset_index(drop=True)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DLL SELECTION RESULTS")
print("=" * 70)

print(
    "\nMI-qualified DLLs:",
    len(mi_qualified)
)

print(
    "Removed due to correlation:",
    len(removed_records)
)

print(
    "Final DLL features:",
    len(remaining)
)

print(
    "Percentage retained:",
    round(
        len(remaining) /
        len(mi_qualified) *
        100,
        2
    ),
    "%"
)

# ============================================================
# REMOVED FEATURES
# ============================================================

removed_df = pd.DataFrame(
    removed_records
)

print("\n" + "=" * 70)
print("REDUNDANT DLLs REMOVED")
print("=" * 70)

if len(removed_df) > 0:

    print(
        removed_df.to_string(index=False)
    )

else:

    print("No highly correlated DLLs found.")

# ============================================================
# FINAL CANDIDATES
# ============================================================

final_dlls = remaining.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("FINAL DLL CANDIDATE SET")
print("=" * 70)

print(
    final_dlls.to_string(index=False)
)

# ============================================================
# SAVE FINAL DLL FEATURES
# ============================================================

FINAL_PATH = "/kaggle/working/DLL_Candidate_Features.csv"

final_dlls.to_csv(
    FINAL_PATH,
    index=False
)

# Save removed features
REMOVED_PATH = (
    "/kaggle/working/"
    "DLL_Removed_Correlated_Features.csv"
)

removed_df.to_csv(
    REMOVED_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("\nMI results:")
print(MI_PATH)

print("\nFinal DLL candidates:")
print(FINAL_PATH)

print("\nCorrelation removal audit:")
print(REMOVED_PATH)

print("\n" + "=" * 70)
print("DLL FEATURE SELECTION COMPLETE")
print("=" * 70)

DLL MI + CORRELATION-REDUCED FEATURE SELECTION

Loading DLL dataset...
Dataset shape: (29498, 631)
Total DLL features: 629

STEP 1: FREQUENCY FILTERING

Frequency threshold: >= 20
DLLs before filtering: 629
DLLs after frequency filtering: 118

STEP 2: MUTUAL INFORMATION
Feature matrix: (29498, 118)
Target: (29498,)

Calculating Mutual Information...

MI results saved to:
/kaggle/working/DLL_Candidate_Features_MI.csv

STEP 3: MUTUAL INFORMATION FILTER
MI threshold: 0.01
Before MI filtering: 118
After MI filtering: 34

MI-qualified DLLs:
                     DLL  MI_Score
             mscoree.dll  0.336208
            kernel32.dll  0.220688
            msvbvm60.dll  0.155394
              user32.dll  0.152095
               gdi32.dll  0.116904
               winmm.dll  0.072743
               mfc42.dll  0.060209
            advapi32.dll  0.054260
             shlwapi.dll  0.052860
              msvcrt.dll  0.052643
            oleaut32.dll  0.052536
            comdlg32.dll  0.041725
   

In [1]:
import pandas as pd

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

print("=" * 70)
print("INSPECTING API DATASET WITHOUT LOADING IT")
print("=" * 70)

# Read only first 5 rows
sample = pd.read_csv(API_PATH, nrows=5)

print("Columns:", len(sample.columns))
print("Shape of sample:", sample.shape)

print("\nFirst 20 columns:")
print(sample.columns[:20].tolist())

print("\nLast 10 columns:")
print(sample.columns[-10:].tolist())

print("\nData types:")
print(sample.dtypes.value_counts())

print("\nFirst rows:")
print(sample.iloc[:, :10])

INSPECTING API DATASET WITHOUT LOADING IT
Columns: 21920
Shape of sample: (5, 21920)

First 20 columns:
['SHA256', 'Type', 'getaclinformation', 'getace', 'getsecuritydescriptordacl', 'regqueryvalueexa', 'regopenkeyexa', 'getsecurityinfo', 'isvalidsid', 'regclosekey', 'getexplicitentriesfromacla', 'getnamedsecurityinfow', 'convertstringsecuritydescriptortosecuritydescriptorw', 'isvalidsecuritydescriptor', 'getsecuritydescriptorgroup', 'regsetvalueexw', 'setsecuritydescriptorsacl', 'getsecuritydescriptorsacl', 'getsecuritydescriptorowner', 'setsecuritydescriptorowner']

Last 10 columns:
['drawdibclose', 'ord1107', 'wsasetblockinghook', 'setupinstallfileexa', 'setupterminatefilelog', 'setuplogfilew', 'setupopenmasterinf', 'setupinstallfileexw', 'setupgetlinecountw', 'setupdigethwprofilefriendlynameexw']

Data types:
int64     21919
object        1
Name: count, dtype: int64

First rows:
                                              SHA256  Type  getaclinformation  \
0  002ce0d28ec990aadbbc

In [2]:
import pandas as pd
import numpy as np
import os
import gc

# ============================================================
# API FREQUENCY ANALYSIS - MEMORY SAFE
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

FREQ_THRESHOLD = 20
CHUNK_SIZE = 1000

print("=" * 70)
print("API FREQUENCY ANALYSIS - MEMORY SAFE")
print("=" * 70)

# ------------------------------------------------------------
# Read header only
# ------------------------------------------------------------

header = pd.read_csv(API_PATH, nrows=0)

ID_COL = "SHA256"
TARGET = "Type"

API_FEATURES = [
    c for c in header.columns
    if c not in [ID_COL, TARGET]
]

print("Total API features:", len(API_FEATURES))
print("Samples expected: 29,498")
print("Frequency threshold:", FREQ_THRESHOLD)

# ------------------------------------------------------------
# Initialize frequency counter
# ------------------------------------------------------------

api_counts = pd.Series(
    0,
    index=API_FEATURES,
    dtype=np.int64
)

# ------------------------------------------------------------
# Process file in chunks
# ------------------------------------------------------------

print("\nProcessing dataset in chunks...")

processed = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=API_FEATURES,
    chunksize=CHUNK_SIZE
):

    # Count samples where API is present
    api_counts += (chunk > 0).sum(axis=0)

    processed += len(chunk)

    if processed % 5000 == 0 or processed >= 29498:
        print(
            f"Processed: {processed:,} / 29,498 samples"
        )

    del chunk
    gc.collect()

# ------------------------------------------------------------
# Frequency results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FREQUENCY RESULTS")
print("=" * 70)

print("Total APIs:", len(api_counts))

print(
    "APIs appearing >= 1 sample:",
    (api_counts >= 1).sum()
)

print(
    "APIs appearing >= 5 samples:",
    (api_counts >= 5).sum()
)

print(
    "APIs appearing >= 10 samples:",
    (api_counts >= 10).sum()
)

print(
    "APIs appearing >= 20 samples:",
    (api_counts >= 20).sum()
)

print(
    "APIs appearing >= 50 samples:",
    (api_counts >= 50).sum()
)

print(
    "APIs appearing >= 100 samples:",
    (api_counts >= 100).sum()
)

print(
    "APIs appearing >= 500 samples:",
    (api_counts >= 500).sum()
)

print(
    "APIs appearing >= 1000 samples:",
    (api_counts >= 1000).sum()
)

# ------------------------------------------------------------
# Frequency statistics
# ------------------------------------------------------------

print("\nFrequency statistics:")

print(api_counts.describe())

# ------------------------------------------------------------
# Top APIs
# ------------------------------------------------------------

frequency_df = pd.DataFrame({
    "API": api_counts.index,
    "Samples_With_API": api_counts.values
})

frequency_df["Percentage"] = (
    frequency_df["Samples_With_API"] /
    processed * 100
)

frequency_df = frequency_df.sort_values(
    "Samples_With_API",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("TOP 30 MOST COMMON APIs")
print("=" * 70)

print(
    frequency_df.head(30).to_string(index=False)
)

# ------------------------------------------------------------
# Rare APIs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAREST APIs")
print("=" * 70)

print(
    frequency_df[
        frequency_df["Samples_With_API"] == 1
    ].head(30).to_string(index=False)
)

# ------------------------------------------------------------
# Save frequency analysis
# ------------------------------------------------------------

FREQ_OUTPUT = "/kaggle/working/API_Frequency_Analysis.csv"

frequency_df.to_csv(
    FREQ_OUTPUT,
    index=False
)

# ------------------------------------------------------------
# Save frequency-qualified APIs
# ------------------------------------------------------------

freq_features = frequency_df[
    frequency_df["Samples_With_API"] >= FREQ_THRESHOLD
]["API"].tolist()

FREQ_FEATURE_OUTPUT = "/kaggle/working/API_Frequency_Selected.csv"

pd.DataFrame({
    "API": freq_features,
    "Samples_With_API": [
        api_counts[x] for x in freq_features
    ]
}).to_csv(
    FREQ_FEATURE_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("FREQUENCY FILTER COMPLETE")
print("=" * 70)

print("Frequency threshold:", FREQ_THRESHOLD)
print("Starting API features:", len(API_FEATURES))
print("Remaining API features:", len(freq_features))

print(
    "Percentage retained:",
    round(
        len(freq_features) /
        len(API_FEATURES) * 100,
        2
    ),
    "%"
)

print("\nSaved:")
print(FREQ_OUTPUT)
print(FREQ_FEATURE_OUTPUT)

# Cleanup
del api_counts
del frequency_df
gc.collect()

API FREQUENCY ANALYSIS - MEMORY SAFE
Total API features: 21918
Samples expected: 29,498
Frequency threshold: 20

Processing dataset in chunks...
Processed: 5,000 / 29,498 samples
Processed: 10,000 / 29,498 samples
Processed: 15,000 / 29,498 samples
Processed: 20,000 / 29,498 samples
Processed: 25,000 / 29,498 samples
Processed: 29,505 / 29,498 samples

FREQUENCY RESULTS
Total APIs: 21918
APIs appearing >= 1 sample: 21918
APIs appearing >= 5 samples: 5879
APIs appearing >= 10 samples: 4094
APIs appearing >= 20 samples: 3079
APIs appearing >= 50 samples: 2374
APIs appearing >= 100 samples: 1911
APIs appearing >= 500 samples: 929
APIs appearing >= 1000 samples: 501

Frequency statistics:
count    21918.000000
mean        93.521352
std        545.252938
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max      13458.000000
dtype: float64

TOP 30 MOST COMMON APIs
                        API  Samples_With_API  Percentage
             getprocaddress     

0

In [3]:
import pandas as pd
import numpy as np
import gc
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# API MUTUAL INFORMATION FEATURE SELECTION
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
FREQ_FEATURE_PATH = "/kaggle/working/API_Frequency_Selected.csv"

MI_THRESHOLD = 0.01
CHUNK_SIZE = 1000

print("=" * 70)
print("API MUTUAL INFORMATION FEATURE SELECTION")
print("=" * 70)

# ============================================================
# LOAD SELECTED API FEATURES
# ============================================================

freq_df = pd.read_csv(FREQ_FEATURE_PATH)

freq_features = freq_df["API"].tolist()

print("\nFrequency-qualified APIs:", len(freq_features))
print("MI threshold:", MI_THRESHOLD)

# ============================================================
# LOAD TARGET + SELECTED FEATURES ONLY
# ============================================================

print("\nLoading selected API features...")

usecols = ["Type"] + freq_features

chunks = []

processed = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=usecols,
    chunksize=CHUNK_SIZE
):

    # Convert API features to compact binary representation
    X_chunk = (
        chunk[freq_features]
        .to_numpy(dtype=np.uint8)
    )

    chunks.append(X_chunk)

    processed += len(chunk)

    if processed % 5000 == 0 or processed >= 29498:
        print(
            f"Processed: {processed:,} / 29,498 samples"
        )

    del chunk
    gc.collect()

# Combine compact chunks
X = np.vstack(chunks)

del chunks
gc.collect()

# Load target separately
target_df = pd.read_csv(
    API_PATH,
    usecols=["Type"]
)

y = target_df["Type"].to_numpy()

del target_df
gc.collect()

print("\n" + "=" * 70)
print("FEATURE MATRIX READY")
print("=" * 70)

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("Approx X memory:",
      round(X.nbytes / (1024**2), 2), "MB")

print("y shape:", y.shape)

# ============================================================
# MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING MUTUAL INFORMATION")
print("=" * 70)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("\nThis may take some time...")

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

# ============================================================
# MI RESULTS
# ============================================================

mi_df = pd.DataFrame({
    "API": freq_features,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("MUTUAL INFORMATION STATISTICS")
print("=" * 70)

print(mi_df["MI_Score"].describe())

# ============================================================
# TOP 30
# ============================================================

print("\n" + "=" * 70)
print("TOP 30 APIs BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(30).to_string(index=False)
)

# ============================================================
# MI THRESHOLD ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("MI THRESHOLD ANALYSIS")
print("=" * 70)

for threshold in [
    0.001,
    0.005,
    0.010,
    0.020,
    0.030,
    0.050,
    0.100
]:

    count = (
        mi_df["MI_Score"] >= threshold
    ).sum()

    print(
        f"MI >= {threshold:<6}: "
        f"{count:4d} APIs"
    )

# ============================================================
# MI FILTER
# ============================================================

mi_selected = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

print("\n" + "=" * 70)
print("MI FILTER RESULT")
print("=" * 70)

print(
    "Before MI filtering:",
    len(mi_df)
)

print(
    "After MI filtering:",
    len(mi_selected)
)

print(
    "Percentage retained:",
    round(
        len(mi_selected) /
        len(mi_df) * 100,
        2
    ),
    "%"
)

# ============================================================
# SAVE
# ============================================================

MI_OUTPUT = "/kaggle/working/API_Candidate_Features_MI.csv"

mi_df.to_csv(
    MI_OUTPUT,
    index=False
)

MI_SELECTED_OUTPUT = "/kaggle/working/API_MI_Selected.csv"

mi_selected.to_csv(
    MI_SELECTED_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("All MI scores:")
print(MI_OUTPUT)

print("\nMI-qualified APIs:")
print(MI_SELECTED_OUTPUT)

# ============================================================
# CLEANUP
# ============================================================

del X
del y
del mi_scores
del mi_df
del mi_selected

gc.collect()

print("\n" + "=" * 70)
print("API MI ANALYSIS COMPLETE")
print("=" * 70)

API MUTUAL INFORMATION FEATURE SELECTION

Frequency-qualified APIs: 3079
MI threshold: 0.01

Loading selected API features...
Processed: 5,000 / 29,498 samples
Processed: 10,000 / 29,498 samples
Processed: 15,000 / 29,498 samples
Processed: 20,000 / 29,498 samples
Processed: 25,000 / 29,498 samples
Processed: 29,505 / 29,498 samples

FEATURE MATRIX READY
X shape: (29505, 3079)
X dtype: uint8
Approx X memory: 86.64 MB
y shape: (29505,)

CALCULATING MUTUAL INFORMATION
Features: 3079
Samples: 29505

This may take some time...

MUTUAL INFORMATION STATISTICS
count    3079.000000
mean        0.016708
std         0.028958
min         0.000480
25%         0.002563
50%         0.006066
75%         0.018802
max         0.336230
Name: MI_Score, dtype: float64

TOP 30 APIs BY MUTUAL INFORMATION
                        API  MI_Score
                 corexemain  0.336230
               virtualalloc  0.184475
                  heapalloc  0.182608
             getprocaddress  0.180877
                

In [1]:
import pandas as pd
import numpy as np
import os

print("=" * 70)
print("API TOP-MI FEATURE REDUCTION")
print("=" * 70)

MI_PATH = "/kaggle/working/API_MI_Selected.csv"

# ============================================================
# LOAD MI RESULTS ONLY
# ============================================================

mi_df = pd.read_csv(MI_PATH)

print("MI-qualified APIs:", len(mi_df))

# Sort by MI
mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# SELECT TOP 200
# ============================================================

TOP_N = 200

top_api_df = mi_df.head(TOP_N).copy()

print("\nTop API selection")
print("-" * 70)

print("Before:", len(mi_df))
print("After :", len(top_api_df))

print("\nTop 30 APIs:")
print(
    top_api_df.head(30).to_string(index=False)
)

# ============================================================
# SAVE
# ============================================================

TOP_API_PATH = "/kaggle/working/API_Top200_MI.csv"

top_api_df.to_csv(
    TOP_API_PATH,
    index=False
)

print("\nSaved:")
print(TOP_API_PATH)

print("\n" + "=" * 70)
print("TOP-MI API REDUCTION COMPLETE")
print("=" * 70)

API TOP-MI FEATURE REDUCTION
MI-qualified APIs: 1188

Top API selection
----------------------------------------------------------------------
Before: 1188
After : 200

Top 30 APIs:
                        API  MI_Score
                 corexemain  0.336230
               virtualalloc  0.184475
                  heapalloc  0.182608
             getprocaddress  0.180877
                  loadicona  0.179674
               loadlibrarya  0.176711
setunhandledexceptionfilter  0.175471
                   heapfree  0.174350
           terminateprocess  0.172306
                 heapcreate  0.171398
             sethandlecount  0.170589
         getcurrentthreadid  0.164780
                   getoemcp  0.161344
                virtualfree  0.161185
                   tlsalloc  0.160850
     getenvironmentstringsw  0.160135
               lcmapstringw  0.159910
        widechartomultibyte  0.159708
    freeenvironmentstringsw  0.159200
             getstringtypew  0.159061
               setla

In [1]:
import pandas as pd
import os

print("=" * 70)
print("VERIFYING SELECTED FEATURE FILES")
print("=" * 70)

files = [
    "/kaggle/working/API_Top200_MI.csv",
    "/kaggle/working/API_MI_Selected.csv",
    "/kaggle/working/DLL_Candidate_Features.csv",
    "/kaggle/working/DLL_Candidate_Features_MI.csv",
]

for path in files:
    print(
        f"{os.path.basename(path):40s}",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

print("\n" + "=" * 70)

api = pd.read_csv(
    "/kaggle/working/API_Top200_MI.csv"
)

dll = pd.read_csv(
    "/kaggle/working/DLL_Candidate_Features.csv"
)

print("Top API features :", len(api))
print("Final DLL features:", len(dll))

print("\nAPI columns:")
print(api.columns.tolist())

print("\nDLL columns:")
print(dll.columns.tolist())

VERIFYING SELECTED FEATURE FILES
API_Top200_MI.csv                        EXISTS
API_MI_Selected.csv                      EXISTS
DLL_Candidate_Features.csv               EXISTS
DLL_Candidate_Features_MI.csv            EXISTS

Top API features : 200
Final DLL features: 27

API columns:
['API', 'MI_Score']

DLL columns:
['DLL', 'MI_Score']


In [2]:
# ============================================================
# MEMORY-SAFE COMPACT API + DLL DATASET BUILDER - FIXED
# ============================================================

import os
import gc
import pandas as pd
import numpy as np

print("=" * 70)
print("BUILDING COMPACT API + DLL DATASET - MEMORY SAFE")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

API_FEATURES_PATH = "/kaggle/working/API_Top200_MI.csv"
DLL_FEATURES_PATH = "/kaggle/working/DLL_Candidate_Features.csv"

API_TEMP = "/kaggle/working/API_Compact_Temp.csv"
FINAL_PATH = "/kaggle/working/API_DLL_Compact_227.csv"

# ============================================================
# LOAD FEATURE LISTS
# ============================================================

api_selected = pd.read_csv(API_FEATURES_PATH)
dll_selected = pd.read_csv(DLL_FEATURES_PATH)

api_features = api_selected["API"].tolist()
dll_features = dll_selected["DLL"].tolist()

print(f"Selected API features : {len(api_features)}")
print(f"Selected DLL features : {len(dll_features)}")
print(f"Total behavioral features : {len(api_features) + len(dll_features)}")

# ============================================================
# VERIFY API COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING API COLUMNS")
print("=" * 70)

api_header = pd.read_csv(API_PATH, nrows=0)

required_api = ["SHA256", "Type"] + api_features

missing_api = [
    col for col in required_api
    if col not in api_header.columns
]

if missing_api:
    print("Missing API columns:")
    print(missing_api)
    raise ValueError("Some selected API columns are missing.")

print("All selected API columns found.")

# ============================================================
# VERIFY DLL COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING DLL COLUMNS")
print("=" * 70)

dll_header = pd.read_csv(DLL_PATH, nrows=0)

required_dll = ["SHA256", "Type"] + dll_features

missing_dll = [
    col for col in required_dll
    if col not in dll_header.columns
]

if missing_dll:
    print("Missing DLL columns:")
    print(missing_dll)
    raise ValueError("Some selected DLL columns are missing.")

print("All selected DLL columns found.")

# ============================================================
# STEP 1: EXTRACT SELECTED API FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: EXTRACTING SELECTED API FEATURES")
print("=" * 70)

if os.path.exists(API_TEMP):
    os.remove(API_TEMP)

api_usecols = ["SHA256", "Type"] + api_features

# IMPORTANT:
# SHA256 = string
# Type = uint8
# API features = uint8

api_dtype = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in api_features:
    api_dtype[feature] = "uint8"

first_chunk = True
total_rows = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=api_usecols,
    chunksize=500,
    dtype=api_dtype,
    low_memory=False
):

    total_rows += len(chunk)

    chunk.to_csv(
        API_TEMP,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    del chunk
    gc.collect()

    if total_rows % 2500 == 0:
        print(f"API processed: {total_rows:,} rows")

print(f"\nAPI extraction complete: {total_rows:,} rows")

# ============================================================
# LOAD COMPACT API DATA
# ============================================================

print("\nLoading compact API dataset...")

api_dtype_load = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in api_features:
    api_dtype_load[feature] = "uint8"

api_df = pd.read_csv(
    API_TEMP,
    dtype=api_dtype_load
)

print("Compact API shape:", api_df.shape)

# ============================================================
# STEP 2: EXTRACT DLL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: EXTRACTING SELECTED DLL FEATURES")
print("=" * 70)

dll_usecols = ["SHA256", "Type"] + dll_features

dll_dtype = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in dll_features:
    dll_dtype[feature] = "uint8"

dll_df = pd.read_csv(
    DLL_PATH,
    usecols=dll_usecols,
    dtype=dll_dtype,
    low_memory=False
)

print("DLL dataset shape:", dll_df.shape)

# ============================================================
# CHECK ROW ALIGNMENT
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DATA ALIGNMENT")
print("=" * 70)

if len(api_df) != len(dll_df):
    raise ValueError(
        f"Row mismatch! API={len(api_df)}, DLL={len(dll_df)}"
    )

print("Row counts match:", len(api_df))

# ============================================================
# CHECK SHA256 ALIGNMENT
# ============================================================

if api_df["SHA256"].equals(dll_df["SHA256"]):

    print("SHA256 order already aligned.")

else:

    print("SHA256 order differs.")
    print("Aligning DLL dataset to API order...")

    dll_df = dll_df.set_index("SHA256")

    api_hashes = api_df["SHA256"]

    dll_df = dll_df.loc[api_hashes].reset_index()

    print("DLL rows successfully aligned.")

# ============================================================
# VERIFY LABEL ALIGNMENT
# ============================================================

if not api_df["Type"].equals(dll_df["Type"]):

    raise ValueError(
        "Type labels do not match between API and DLL datasets."
    )

print("Target labels match.")

# ============================================================
# STEP 3: COMBINE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: COMBINING API + DLL FEATURES")
print("=" * 70)

final_df = pd.concat(
    [
        api_df[["SHA256", "Type"] + api_features],
        dll_df[dll_features]
    ],
    axis=1
)

print("Final shape:", final_df.shape)

# ============================================================
# VERIFY FINAL DATASET
# ============================================================

expected_columns = (
    2
    + len(api_features)
    + len(dll_features)
)

print("\nExpected columns:", expected_columns)
print("Actual columns  :", final_df.shape[1])

if final_df.shape[1] != expected_columns:
    raise ValueError("Final column count mismatch.")

# ============================================================
# VERIFY TARGET
# ============================================================

print("\nTarget distribution:")

print(
    final_df["Type"]
    .value_counts()
    .sort_index()
)

# ============================================================
# VERIFY FEATURE VALUES
# ============================================================

print("\nChecking behavioral feature values...")

feature_columns = api_features + dll_features

unique_values = set(
    final_df[feature_columns]
    .stack()
    .unique()
)

print("Unique behavioral values:", sorted(unique_values))

if not unique_values.issubset({0, 1}):
    raise ValueError(
        "Unexpected values detected. "
        "Behavioral features should be binary 0/1."
    )

print("All behavioral features are binary.")

# ============================================================
# MEMORY
# ============================================================

memory_mb = (
    final_df.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"\nFinal dataframe memory: {memory_mb:.2f} MB")

# ============================================================
# SAVE
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINAL DATASET")
print("=" * 70)

final_df.to_csv(
    FINAL_PATH,
    index=False
)

print("\nSaved successfully:")
print(FINAL_PATH)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("COMPACT DATASET CREATED SUCCESSFULLY")
print("=" * 70)

print(f"Samples              : {len(final_df):,}")
print(f"API features         : {len(api_features)}")
print(f"DLL features         : {len(dll_features)}")
print(f"Behavioral features  : {len(feature_columns)}")
print(f"Total columns        : {len(final_df.columns)}")

print("\nFinal shape:")
print(final_df.shape)

print("\nTarget distribution:")
print(final_df["Type"].value_counts().sort_index())

print("\nOutput:")
print(FINAL_PATH)

BUILDING COMPACT API + DLL DATASET - MEMORY SAFE
Selected API features : 200
Selected DLL features : 27
Total behavioral features : 227

VERIFYING API COLUMNS
All selected API columns found.

VERIFYING DLL COLUMNS
All selected DLL columns found.

STEP 1: EXTRACTING SELECTED API FEATURES
API processed: 2,500 rows
API processed: 5,000 rows
API processed: 7,500 rows
API processed: 10,000 rows
API processed: 12,500 rows
API processed: 15,000 rows
API processed: 17,500 rows
API processed: 20,000 rows
API processed: 22,500 rows
API processed: 25,000 rows
API processed: 27,500 rows

API extraction complete: 29,505 rows

Loading compact API dataset...
Compact API shape: (29505, 202)

STEP 2: EXTRACTING SELECTED DLL FEATURES
DLL dataset shape: (29498, 29)

CHECKING DATA ALIGNMENT


ValueError: Row mismatch! API=29505, DLL=29498

In [3]:
# ============================================================
# DIAGNOSING API / DLL ROW MISMATCH
# ============================================================

import pandas as pd

API_TEMP = "/kaggle/working/API_Compact_Temp.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

print("=" * 70)
print("DIAGNOSING API / DLL ROW MISMATCH")
print("=" * 70)

# ------------------------------------------------------------
# LOAD ONLY SHA256 + TYPE
# ------------------------------------------------------------

print("\nLoading API identifiers...")

api_ids = pd.read_csv(
    API_TEMP,
    usecols=["SHA256", "Type"],
    dtype={
        "SHA256": "string",
        "Type": "uint8"
    }
)

print("API rows:", len(api_ids))

print("\nLoading DLL identifiers...")

dll_ids = pd.read_csv(
    DLL_PATH,
    usecols=["SHA256", "Type"],
    dtype={
        "SHA256": "string",
        "Type": "uint8"
    }
)

print("DLL rows:", len(dll_ids))

# ============================================================
# DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DUPLICATES")
print("=" * 70)

api_duplicates = api_ids["SHA256"].duplicated().sum()
dll_duplicates = dll_ids["SHA256"].duplicated().sum()

print("API duplicate SHA256:", api_duplicates)
print("DLL duplicate SHA256:", dll_duplicates)

# ============================================================
# SET COMPARISON
# ============================================================

api_hashes = set(api_ids["SHA256"])
dll_hashes = set(dll_ids["SHA256"])

api_only = api_hashes - dll_hashes
dll_only = dll_hashes - api_hashes

print("\n" + "=" * 70)
print("SHA256 SET COMPARISON")
print("=" * 70)

print("Unique API hashes:", len(api_hashes))
print("Unique DLL hashes:", len(dll_hashes))

print("API-only hashes:", len(api_only))
print("DLL-only hashes:", len(dll_only))

# ============================================================
# SHOW API-ONLY SAMPLES
# ============================================================

if len(api_only) > 0:

    print("\n" + "=" * 70)
    print("API-ONLY SAMPLES")
    print("=" * 70)

    api_only_df = api_ids[
        api_ids["SHA256"].isin(api_only)
    ].copy()

    print(api_only_df.to_string(index=False))

# ============================================================
# SHOW DLL-ONLY SAMPLES
# ============================================================

if len(dll_only) > 0:

    print("\n" + "=" * 70)
    print("DLL-ONLY SAMPLES")
    print("=" * 70)

    dll_only_df = dll_ids[
        dll_ids["SHA256"].isin(dll_only)
    ].copy()

    print(dll_only_df.to_string(index=False))

# ============================================================
# LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTIONS")
print("=" * 70)

print("\nAPI:")
print(api_ids["Type"].value_counts().sort_index())

print("\nDLL:")
print(dll_ids["Type"].value_counts().sort_index())

# ============================================================
# FINAL DIAGNOSIS
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSIS")
print("=" * 70)

if len(api_only) == 7 and len(dll_only) == 0:
    print("Exactly 7 API-only samples found.")
    print("DLL dataset is a subset of API dataset.")

elif len(api_only) == 0 and len(dll_only) == 0:
    print("SHA256 sets match despite row-count difference.")
    print("Likely duplicate SHA256 rows exist in one dataset.")

else:
    print("API and DLL datasets contain different samples.")
    print("We need to align using SHA256 intersection.")

print("\nDiagnosis complete.")

DIAGNOSING API / DLL ROW MISMATCH

Loading API identifiers...
API rows: 29505

Loading DLL identifiers...
DLL rows: 29498

CHECKING DUPLICATES
API duplicate SHA256: 5
DLL duplicate SHA256: 3

SHA256 SET COMPARISON
Unique API hashes: 29500
Unique DLL hashes: 29495
API-only hashes: 8
DLL-only hashes: 3

API-ONLY SAMPLES
                                                          SHA256  Type
0a67ff35add751a754db8f7d55515b0dbac034b87274cfd1cc351f7679a98b2f     5
4e206dcf64691be28bb9194c7975c4fa84a10ce8565d518939dfd7b95103b0d0     5
a18de3e6fcd33b24740e042b105abf4c1bd06d7cf904a4ac83c25a4140431426     5
ca4896bca693d072c950104571b902964e437bcf92be959262573d6147291139     5
f2725ad45e1ca66ad1380ce1e59e43b9793c4cdfdba0eb023aba15ce7a4e102e     5
323b929d2450e6f1bdf0bb517d5a2e1bbbf7b3b43b06737bd97e9b74f9bee926     5
cb64fe8950ae788d699380f0676a21aa6ba2329f8055309fd51268d9c1e0c8b4     5
9084e52dcbc7f1ae40e2e077114f70477d2954cf149cabde5c83439f75ff13cc     5

DLL-ONLY SAMPLES
                       

In [4]:
# ============================================================
# BUILD FINAL ALIGNED API + DLL DATASET
# SHA256-BASED ALIGNMENT - MEMORY SAFE
# ============================================================

import pandas as pd
import os
import gc

print("=" * 70)
print("BUILDING FINAL ALIGNED API + DLL DATASET")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

API_FEATURE_PATH = "/kaggle/working/API_Top200_MI.csv"
DLL_FEATURE_PATH = "/kaggle/working/DLL_Candidate_Features.csv"

OUTPUT_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

# ============================================================
# LOAD SELECTED FEATURE LISTS
# ============================================================

api_features = pd.read_csv(API_FEATURE_PATH)["API"].tolist()
dll_features = pd.read_csv(DLL_FEATURE_PATH)["DLL"].tolist()

print(f"\nSelected API features : {len(api_features)}")
print(f"Selected DLL features : {len(dll_features)}")
print(f"Total behavioral features : {len(api_features) + len(dll_features)}")

# ============================================================
# STEP 1: LOAD DLL DATASET
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: LOADING DLL DATASET")
print("=" * 70)

dll_usecols = ["SHA256", "Type"] + dll_features

dll_df = pd.read_csv(
    DLL_PATH,
    usecols=dll_usecols
)

print(f"DLL rows loaded: {len(dll_df):,}")

# ============================================================
# CHECK DUPLICATES
# ============================================================

print("\nChecking DLL duplicates...")

dll_duplicates = dll_df["SHA256"].duplicated().sum()

print(f"DLL duplicate SHA256 rows: {dll_duplicates}")

# Keep first occurrence
dll_df = dll_df.drop_duplicates(
    subset="SHA256",
    keep="first"
).reset_index(drop=True)

print(f"DLL unique samples: {len(dll_df):,}")

# ============================================================
# STEP 2: CREATE DLL LOOKUP
# ============================================================

print("\nCreating DLL SHA256 lookup...")

dll_hashes = set(dll_df["SHA256"])

print(f"DLL unique hashes: {len(dll_hashes):,}")

# ============================================================
# STEP 3: STREAM API DATASET
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: STREAMING API DATASET")
print("=" * 70)

api_usecols = ["SHA256", "Type"] + api_features

matched_chunks = []

total_rows = 0
matched_rows = 0
api_seen = set()

for chunk in pd.read_csv(
    API_PATH,
    usecols=api_usecols,
    dtype={
        "SHA256": "string",
        "Type": "int8"
    },
    chunksize=2500
):

    total_rows += len(chunk)

    # Remove duplicate SHA256 within API dataset
    chunk = chunk[
        ~chunk["SHA256"].isin(api_seen)
    ].copy()

    # Add hashes to seen set
    api_seen.update(chunk["SHA256"].tolist())

    # Keep only samples present in DLL dataset
    chunk = chunk[
        chunk["SHA256"].isin(dll_hashes)
    ].copy()

    if len(chunk) > 0:
        matched_rows += len(chunk)
        matched_chunks.append(chunk)

    if total_rows % 10000 < 2500:
        print(
            f"Processed: {total_rows:,} | "
            f"Matched: {matched_rows:,}"
        )

print("\nAPI streaming complete.")

# ============================================================
# COMBINE API MATCHES
# ============================================================

api_df = pd.concat(
    matched_chunks,
    ignore_index=True
)

del matched_chunks
gc.collect()

print(f"\nAPI matched unique samples: {len(api_df):,}")

# ============================================================
# STEP 4: MERGE API + DLL USING SHA256
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: MERGING API + DLL DATA")
print("=" * 70)

# Rename Type columns
api_df = api_df.rename(
    columns={"Type": "Type_API"}
)

dll_df = dll_df.rename(
    columns={"Type": "Type_DLL"}
)

# Inner join using SHA256
final_df = api_df.merge(
    dll_df,
    on="SHA256",
    how="inner",
    suffixes=("", "_DLL")
)

print(f"Final merged shape: {final_df.shape}")

# ============================================================
# STEP 5: VERIFY LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: VERIFYING LABEL CONSISTENCY")
print("=" * 70)

label_mismatch = (
    final_df["Type_API"] != final_df["Type_DLL"]
).sum()

print(f"Label mismatches: {label_mismatch}")

if label_mismatch > 0:

    print("\nWARNING: Label mismatch detected!")

    mismatch_df = final_df[
        final_df["Type_API"] != final_df["Type_DLL"]
    ][
        ["SHA256", "Type_API", "Type_DLL"]
    ]

    print(mismatch_df.head(20))

else:
    print("All API and DLL labels match.")

# ============================================================
# STEP 6: CREATE FINAL TARGET
# ============================================================

final_df["Type"] = final_df["Type_API"]

# Remove duplicate target columns
final_df = final_df.drop(
    columns=["Type_API", "Type_DLL"]
)

# ============================================================
# STEP 7: REMOVE ANY REMAINING DUPLICATES
# ============================================================

before = len(final_df)

final_df = final_df.drop_duplicates(
    subset="SHA256",
    keep="first"
).reset_index(drop=True)

after = len(final_df)

print(f"\nDuplicate samples removed: {before - after}")
print(f"Final unique samples: {after:,}")

# ============================================================
# STEP 8: VERIFY FEATURE COUNT
# ============================================================

expected_features = (
    1 +                 # SHA256
    len(api_features) +
    len(dll_features) +
    1                   # Type
)

print("\n" + "=" * 70)
print("FINAL DATASET VERIFICATION")
print("=" * 70)

print(f"Expected columns : {expected_features}")
print(f"Actual columns   : {len(final_df.columns)}")

print(f"\nSHA256 column present: {'SHA256' in final_df.columns}")
print(f"Type column present  : {'Type' in final_df.columns}")

print(
    f"API features present : "
    f"{sum(x in final_df.columns for x in api_features)} / {len(api_features)}"
)

print(
    f"DLL features present : "
    f"{sum(x in final_df.columns for x in dll_features)} / {len(dll_features)}"
)

# ============================================================
# TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("FINAL TARGET DISTRIBUTION")
print("=" * 70)

print(final_df["Type"].value_counts().sort_index())

# ============================================================
# SAVE
# ============================================================

print("\nSaving final dataset...")

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved to:")
print(OUTPUT_PATH)

print("\n" + "=" * 70)
print("FINAL COMPACT DATASET COMPLETE")
print("=" * 70)

print(f"Final shape: {final_df.shape}")
print(f"Behavioral features: {len(api_features) + len(dll_features)}")
print(f"Samples: {len(final_df):,}")

BUILDING FINAL ALIGNED API + DLL DATASET

Selected API features : 200
Selected DLL features : 27
Total behavioral features : 227

STEP 1: LOADING DLL DATASET
DLL rows loaded: 29,498

Checking DLL duplicates...
DLL duplicate SHA256 rows: 3
DLL unique samples: 29,495

Creating DLL SHA256 lookup...
DLL unique hashes: 29,495

STEP 3: STREAMING API DATASET
Processed: 10,000 | Matched: 10,000
Processed: 20,000 | Matched: 20,000

API streaming complete.

API matched unique samples: 29,495

STEP 4: MERGING API + DLL DATA
Final merged shape: (29495, 230)

STEP 5: VERIFYING LABEL CONSISTENCY
Label mismatches: 0
All API and DLL labels match.

Duplicate samples removed: 3
Final unique samples: 29,492

FINAL DATASET VERIFICATION
Expected columns : 229
Actual columns   : 229

SHA256 column present: True
Type column present  : True
API features present : 200 / 200
DLL features present : 27 / 27

FINAL TARGET DISTRIBUTION
Type
0    1877
1    5022
2    4643
3    4957
4    5076
5    4218
6    3699
Name:

In [5]:
# ============================================================
# FINAL SANITY CHECK - COMPACT API + DLL DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

print("=" * 70)
print("FINAL DATASET SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print(f"\nDataset shape: {df.shape}")

# ------------------------------------------------------------
# BASIC STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. BASIC STRUCTURE")
print("=" * 70)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {len(df.columns):,}")
print(f"Memory MB : {df.memory_usage(deep=True).sum() / 1024**2:.2f}")

print(f"\nSHA256 present: {'SHA256' in df.columns}")
print(f"Type present  : {'Type' in df.columns}")

behavioral_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

print(f"Behavioral features: {len(behavioral_cols)}")

# ------------------------------------------------------------
# MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. MISSING VALUES")
print("=" * 70)

missing_total = df[behavioral_cols].isna().sum().sum()

print(f"Total missing behavioral values: {missing_total:,}")

if missing_total == 0:
    print("PASS: No missing behavioral values.")
else:
    print("WARNING: Missing values detected.")

# ------------------------------------------------------------
# DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. DATA TYPES")
print("=" * 70)

print(df[behavioral_cols].dtypes.value_counts())

# ------------------------------------------------------------
# UNIQUE VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. FEATURE VALUE CHECK")
print("=" * 70)

non_binary = []

for col in behavioral_cols:
    values = df[col].dropna().unique()

    if not set(values).issubset({0, 1}):
        non_binary.append(
            (col, len(values), sorted(values)[:10])
        )

print(f"Total behavioral features : {len(behavioral_cols)}")
print(f"Binary features            : {len(behavioral_cols) - len(non_binary)}")
print(f"Non-binary features        : {len(non_binary)}")

if non_binary:
    print("\nNon-binary examples:")
    for item in non_binary[:20]:
        print(item)
else:
    print("PASS: All behavioral features are binary.")

# ------------------------------------------------------------
# ZERO VARIANCE / CONSTANT FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. CONSTANT FEATURES")
print("=" * 70)

nunique = df[behavioral_cols].nunique()

constant_features = nunique[nunique <= 1]

print(f"Constant features: {len(constant_features)}")

if len(constant_features) > 0:
    print("\nConstant features:")
    print(constant_features)
else:
    print("PASS: No constant features.")

# ------------------------------------------------------------
# FEATURE FREQUENCY / SPARSITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. FEATURE SPARSITY")
print("=" * 70)

feature_frequency = df[behavioral_cols].sum()

sparsity = (
    1 - feature_frequency / len(df)
) * 100

print(
    f"Average feature presence: "
    f"{feature_frequency.mean():.2f} samples"
)

print(
    f"Average sparsity: "
    f"{sparsity.mean():.2f}%"
)

print("\nMost common behavioral features:")

top_features = pd.DataFrame({
    "Feature": feature_frequency.index,
    "Samples_With_Feature": feature_frequency.values,
    "Percentage": (
        feature_frequency.values / len(df) * 100
    )
}).sort_values(
    "Samples_With_Feature",
    ascending=False
)

print(top_features.head(20).to_string(index=False))

# ------------------------------------------------------------
# DUPLICATE SHA256
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. SHA256 DUPLICATES")
print("=" * 70)

sha_duplicates = df["SHA256"].duplicated().sum()

print(f"Duplicate SHA256 rows: {sha_duplicates}")

# ------------------------------------------------------------
# DUPLICATE BEHAVIORAL VECTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. DUPLICATE BEHAVIORAL VECTORS")
print("=" * 70)

duplicate_vectors = df.duplicated(
    subset=behavioral_cols
).sum()

print(
    f"Duplicate behavioral vectors: "
    f"{duplicate_vectors:,}"
)

# ------------------------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. TARGET DISTRIBUTION")
print("=" * 70)

target_counts = df["Type"].value_counts().sort_index()

target_percent = (
    target_counts / len(df) * 100
).round(2)

target_summary = pd.DataFrame({
    "Samples": target_counts,
    "Percentage": target_percent
})

print(target_summary)

# ------------------------------------------------------------
# CLASS IMBALANCE
# ------------------------------------------------------------

max_class = target_counts.max()
min_class = target_counts.min()

imbalance_ratio = max_class / min_class

print(
    f"\nLargest class / smallest class ratio: "
    f"{imbalance_ratio:.2f}"
)

# ------------------------------------------------------------
# TARGET LEAKAGE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. TARGET LEAKAGE CHECK")
print("=" * 70)

# Correlation of binary features with target
# This is only a rough screening, not proof of leakage.

target_corr = (
    df[behavioral_cols]
    .corrwith(df["Type"])
    .abs()
    .sort_values(ascending=False)
)

print("Top 20 feature-target correlations:")

print(
    target_corr.head(20).to_string()
)

# ------------------------------------------------------------
# SHA256 UNIQUENESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("11. SHA256 UNIQUENESS")
print("=" * 70)

unique_sha = df["SHA256"].nunique()

print(f"Unique SHA256: {unique_sha:,}")
print(f"Total rows   : {len(df):,}")

if unique_sha == len(df):
    print("PASS: Every sample has a unique SHA256.")
else:
    print("WARNING: Duplicate SHA256 values exist.")

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SANITY CHECK SUMMARY")
print("=" * 70)

checks = {
    "Correct row count": len(df) > 29000,
    "229 columns": len(df.columns) == 229,
    "227 behavioral features": len(behavioral_cols) == 227,
    "No missing values": missing_total == 0,
    "All features binary": len(non_binary) == 0,
    "No constant features": len(constant_features) == 0,
    "Unique SHA256": unique_sha == len(df),
}

for name, result in checks.items():
    print(
        f"{'PASS' if result else 'FAIL'} : {name}"
    )

print("\n" + "=" * 70)
print("SANITY CHECK COMPLETE")
print("=" * 70)

FINAL DATASET SANITY CHECK

Dataset shape: (29492, 229)

1. BASIC STRUCTURE
Rows      : 29,492
Columns   : 229
Memory MB : 54.48

SHA256 present: True
Type present  : True
Behavioral features: 227

2. MISSING VALUES
Total missing behavioral values: 0
PASS: No missing behavioral values.

3. DATA TYPES
int64    227
Name: count, dtype: int64

4. FEATURE VALUE CHECK
Total behavioral features : 227
Binary features            : 227
Non-binary features        : 0
PASS: All behavioral features are binary.

5. CONSTANT FEATURES
Constant features: 0
PASS: No constant features.

6. FEATURE SPARSITY
Average feature presence: 4171.24 samples
Average sparsity: 85.86%

Most common behavioral features:
                 Feature  Samples_With_Feature  Percentage
            kernel32.dll                 14595   49.487997
          getprocaddress                 13454   45.619151
             mscoree.dll                 12370   41.943578
              corexemain                 12363   41.919843
         